# Indexing: `loc` vs `iloc` (the skill interviewers probe)

- `loc` selects by **label** (the index value).
- `iloc` selects by **integer position** (0-based row number).

They often return the same thing — until the index isn't `0..n-1`. That difference causes the most common pandas bugs.

In [1]:
import pandas as pd

In [2]:
# Local execution already has the data files.
# On Colab (which opens just this notebook) we download them:
import os
import urllib.request

NEEDS_FILES = ['jamesbond.csv']

for _f in NEEDS_FILES:
    if not os.path.exists(_f):
        urllib.request.urlretrieve('https://raw.githubusercontent.com/sivabalanb/Data-Analysis-with-Pandas-and-Python/master/' + _f, _f)
        print('downloaded', _f)


## `set_index` — choosing the row labels

`set_index("Film", inplace=True)` makes the Film column the row labels. Now rows are addressable by movie title instead of position.

In [3]:
bond = pd.read_csv("jamesbond.csv")
bond

,Film,Year,Actor,Director,Box Office,Budget,Bond Actor Salary
0,Dr. No,1962,Sean Connery,Terence Young,448.8,7.0,0.6
1,From Russia with Love,1963,Sean Connery,Terence Young,543.8,12.6,1.6
2,Goldfinger,1964,Sean Connery,Guy Hamilton,820.4,18.6,3.2
3,Thunderball,1965,Sean Connery,Terence Young,848.1,41.9,4.7
4,Casino Royale,1967,David Niven,Ken Hughes,315.0,85.0,NaN
5,You Only Live Twice,1967,Sean Connery,Lewis Gilbert,514.2,59.9,4.4
6,On Her Majesty's Secret Service,1969,George Lazenby,Peter R. Hunt,291.5,37.3,0.6
7,Diamonds Are Forever,1971,Sean Connery,Guy Hamilton,442.5,34.7,5.8
8,Live and Let Die,1973,Roger Moore,Guy Hamilton,460.3,30.8,NaN
9,The Man with the Golden Gun,1974,Roger Moore,Guy Hamilton,334.0,27.7,NaN


In [4]:
bond.set_index("Film", inplace= True)

In [5]:
bond.loc["Goldfinger"]

Year                         1964
Actor                Sean Connery
Director             Guy Hamilton
Box Office                  820.4
Budget                       18.6
Bond Actor Salary             3.2
Name: Goldfinger, dtype: object

## `loc[15]` vs `iloc[15]`

The index is still the default `0..n-1` here, so both return the same row. The real difference appears once the index is non-integer (next cells).

In [6]:
bond = pd.read_csv("jamesbond.csv")

bond.loc[15]
bond.iloc[15]

Film                 A View to a Kill
Year                             1985
Actor                     Roger Moore
Director                    John Glen
Box Office                      275.2
Budget                           54.5
Bond Actor Salary                 9.1
Name: 15, dtype: object

In [7]:
bond.set_index("Film", inplace= True)
bond.sort_index(inplace = True)

## Mixed index demo

After `set_index("Film")` the index is movie titles. Notice:
- `bond.loc["Octopussy"]` finds by label.
- `bond.iloc[14]` finds by *position* — it does not care about labels.

Run both on a *sorted* index and compare which stays stable.

In [8]:
bond.iloc[14]

Year                        1983
Actor                Roger Moore
Director               John Glen
Box Office                 373.8
Budget                      53.9
Bond Actor Salary            7.8
Name: Octopussy, dtype: object

In [9]:
bond.loc["Octopussy"]

Year                        1983
Actor                Roger Moore
Director               John Glen
Box Office                 373.8
Budget                      53.9
Bond Actor Salary            7.8
Name: Octopussy, dtype: object

In [10]:
bond.loc["Octopussy"]

Year                        1983
Actor                Roger Moore
Director               John Glen
Box Office                 373.8
Budget                      53.9
Bond Actor Salary            7.8
Name: Octopussy, dtype: object

In [11]:
bond.iloc[14]

Year                        1983
Actor                Roger Moore
Director               John Glen
Box Office                 373.8
Budget                      53.9
Bond Actor Salary            7.8
Name: Octopussy, dtype: object

## Selecting rows *and* columns

`loc[row_label, [column_labels]]` and `iloc[row_pos, [column_positions]]` — combine row and column selection in one call.

In [12]:
bond.loc["Octopussy",["Actor","Year"]]

Actor    Roger Moore
Year            1983
Name: Octopussy, dtype: object

In [13]:
bond.iloc[14,[1,2,4]]

Actor       Roger Moore
Director      John Glen
Budget             53.9
Name: Octopussy, dtype: object

## Setting values and the copy warning

`bond.iloc[14, [1]] = "..."` modifies a value in place.

The following cells demonstrate a classic bug: chained assignment (`df[mask]["col"] = ...`) that silently does nothing. `df2[mask]["Actor"] = ...` is a chain — pandas often can't write through it. **Always use `.loc[mask, "col"]` to modify filtered rows.**

In [14]:
bond.iloc[14,[1]] = "Sir Roger Moore"

In [15]:
bond["Actor"].unique()

<StringArray>
[    'Roger Moore',    'Daniel Craig',     'David Niven',    'Sean Connery',
  'Pierce Brosnan',  'Timothy Dalton', 'Sir Roger Moore',  'George Lazenby']
Length: 8, dtype: str

In [16]:
mask = bond["Actor"] == "Sean Connery"

In [17]:
bond[mask]["Actor"] = "Sir Sean Connery"

/tmp/ipykernel_42600/2959481625.py:1: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment.
Such chained assignment never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

Try using '.loc[row_indexer, col_indexer] = value' instead, to perform the assignment in a single step.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html#chained-assignment
  bond[mask]["Actor"] = "Sir Sean Connery"


In [18]:
df2 = bond["Actor"]
df2[mask]["Actor"] = "Sir Sean Connery"

/tmp/ipykernel_42600/3328428467.py:2: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment.
Such chained assignment never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

Try using '.loc[row_indexer, col_indexer] = value' instead, to perform the assignment in a single step.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html#chained-assignment
  df2[mask]["Actor"] = "Sir Sean Connery"


In [19]:
df2

Film
A View to a Kill                       Roger Moore
Casino Royale                         Daniel Craig
Casino Royale                          David Niven
Diamonds Are Forever                  Sean Connery
Die Another Day                     Pierce Brosnan
Dr. No                                Sean Connery
For Your Eyes Only                     Roger Moore
From Russia with Love                 Sean Connery
GoldenEye                           Pierce Brosnan
Goldfinger                            Sean Connery
Licence to Kill                     Timothy Dalton
Live and Let Die                       Roger Moore
Moonraker                              Roger Moore
Never Say Never Again                 Sean Connery
Octopussy                          Sir Roger Moore
On Her Majesty's Secret Service     George Lazenby
Quantum of Solace                     Daniel Craig
Skyfall                               Daniel Craig
Spectre                               Daniel Craig
The Living Daylights      

## Renaming columns

`rename(columns={...})` changes column names without touching data. Note that `rename` returns a *new* DataFrame by default — assign it back or use `inplace=True`.

In [20]:
bond = pd.read_csv("jamesbond.csv")
bond.rename(columns={"Year":"Release Date"})

,Film,Release Date,Actor,Director,Box Office,Budget,Bond Actor Salary
0,Dr. No,1962,Sean Connery,Terence Young,448.8,7.0,0.6
1,From Russia with Love,1963,Sean Connery,Terence Young,543.8,12.6,1.6
2,Goldfinger,1964,Sean Connery,Guy Hamilton,820.4,18.6,3.2
3,Thunderball,1965,Sean Connery,Terence Young,848.1,41.9,4.7
4,Casino Royale,1967,David Niven,Ken Hughes,315.0,85.0,NaN
5,You Only Live Twice,1967,Sean Connery,Lewis Gilbert,514.2,59.9,4.4
6,On Her Majesty's Secret Service,1969,George Lazenby,Peter R. Hunt,291.5,37.3,0.6
7,Diamonds Are Forever,1971,Sean Connery,Guy Hamilton,442.5,34.7,5.8
8,Live and Let Die,1973,Roger Moore,Guy Hamilton,460.3,30.8,NaN
9,The Man with the Golden Gun,1974,Roger Moore,Guy Hamilton,334.0,27.7,NaN


In [21]:
bond.columns

Index(['Film', 'Year', 'Actor', 'Director', 'Box Office', 'Budget',
       'Bond Actor Salary'],
      dtype='str')

## `drop` — removing rows and columns

`drop(..., axis=0)` removes rows (by label), `drop(..., axis=1)` removes columns.

In [22]:
bond = bond.set_index("Film")
bond.drop("Dr. No", axis=0)
bond.drop("Box Office", axis = 1)

,Year,Actor,Director,Budget,Bond Actor Salary
Film,,,,,
Dr. No,1962,Sean Connery,Terence Young,7.0,0.6
From Russia with Love,1963,Sean Connery,Terence Young,12.6,1.6
Goldfinger,1964,Sean Connery,Guy Hamilton,18.6,3.2
Thunderball,1965,Sean Connery,Terence Young,41.9,4.7
Casino Royale,1967,David Niven,Ken Hughes,85.0,NaN
You Only Live Twice,1967,Sean Connery,Lewis Gilbert,59.9,4.4
On Her Majesty's Secret Service,1969,George Lazenby,Peter R. Hunt,37.3,0.6
Diamonds Are Forever,1971,Sean Connery,Guy Hamilton,34.7,5.8
Live and Let Die,1973,Roger Moore,Guy Hamilton,30.8,NaN


In [23]:
bond = pd.read_csv("jamesbond.csv")


## `sample` — random rows

`sample()` picks random rows — great for eyeballing data. `sample(5)` = fixed count, `sample(frac=0.25)` = 25% of rows.

In [24]:
bond.sample()

,Film,Year,Actor,Director,Box Office,Budget,Bond Actor Salary
23,Quantum of Solace,2008,Daniel Craig,Marc Forster,514.2,181.4,8.1


In [25]:
bond.sample(5)

,Film,Year,Actor,Director,Box Office,Budget,Bond Actor Salary
9,The Man with the Golden Gun,1974,Roger Moore,Guy Hamilton,334.0,27.7,NaN
18,GoldenEye,1995,Pierce Brosnan,Martin Campbell,518.5,76.9,5.1
15,A View to a Kill,1985,Roger Moore,John Glen,275.2,54.5,9.1
6,On Her Majesty's Secret Service,1969,George Lazenby,Peter R. Hunt,291.5,37.3,0.6
16,The Living Daylights,1987,Timothy Dalton,John Glen,313.5,68.8,5.2


bond.sample()

In [26]:
bond.sample(frac= .25)

,Film,Year,Actor,Director,Box Office,Budget,Bond Actor Salary
22,Casino Royale,2006,Daniel Craig,Martin Campbell,581.5,145.3,3.3
23,Quantum of Solace,2008,Daniel Craig,Marc Forster,514.2,181.4,8.1
10,The Spy Who Loved Me,1977,Roger Moore,Lewis Gilbert,533.0,45.1,NaN
2,Goldfinger,1964,Sean Connery,Guy Hamilton,820.4,18.6,3.2
8,Live and Let Die,1973,Roger Moore,Guy Hamilton,460.3,30.8,NaN
4,Casino Royale,1967,David Niven,Ken Hughes,315.0,85.0,NaN


## Top N rows

`nlargest(4, "Box Office")` returns the 4 rows with the highest values in that column. (There's also `nsmallest`.)

In [27]:
bond.nlargest(n = 4 , columns= "Box Office")

,Film,Year,Actor,Director,Box Office,Budget,Bond Actor Salary
24,Skyfall,2012,Daniel Craig,Sam Mendes,943.5,170.2,14.5
3,Thunderball,1965,Sean Connery,Terence Young,848.1,41.9,4.7
2,Goldfinger,1964,Sean Connery,Guy Hamilton,820.4,18.6,3.2
25,Spectre,2015,Daniel Craig,Sam Mendes,726.7,206.3,NaN


In [28]:
bond = pd.read_csv("jamesbond.csv")



In [29]:
mask = bond["Actor"] == "Sean Connery"
bond[mask]

,Film,Year,Actor,Director,Box Office,Budget,Bond Actor Salary
0,Dr. No,1962,Sean Connery,Terence Young,448.8,7.0,0.6
1,From Russia with Love,1963,Sean Connery,Terence Young,543.8,12.6,1.6
2,Goldfinger,1964,Sean Connery,Guy Hamilton,820.4,18.6,3.2
3,Thunderball,1965,Sean Connery,Terence Young,848.1,41.9,4.7
5,You Only Live Twice,1967,Sean Connery,Lewis Gilbert,514.2,59.9,4.4
7,Diamonds Are Forever,1971,Sean Connery,Guy Hamilton,442.5,34.7,5.8
13,Never Say Never Again,1983,Sean Connery,Irvin Kershner,380.0,86.0,NaN


## `where` — conditional masking

`df.where(condition)` keeps the values where the condition is True and fills everything else with `NaN`. It's `df[mask]` but with *shape preserved*.

In [30]:
bond.where(bond["Actor"] == "Sean Connery")

bond.where(bond["Box Office"] > 100)

,Film,Year,Actor,Director,Box Office,Budget,Bond Actor Salary
0,Dr. No,1962,Sean Connery,Terence Young,448.8,7.0,0.6
1,From Russia with Love,1963,Sean Connery,Terence Young,543.8,12.6,1.6
2,Goldfinger,1964,Sean Connery,Guy Hamilton,820.4,18.6,3.2
3,Thunderball,1965,Sean Connery,Terence Young,848.1,41.9,4.7
4,Casino Royale,1967,David Niven,Ken Hughes,315.0,85.0,NaN
5,You Only Live Twice,1967,Sean Connery,Lewis Gilbert,514.2,59.9,4.4
6,On Her Majesty's Secret Service,1969,George Lazenby,Peter R. Hunt,291.5,37.3,0.6
7,Diamonds Are Forever,1971,Sean Connery,Guy Hamilton,442.5,34.7,5.8
8,Live and Let Die,1973,Roger Moore,Guy Hamilton,460.3,30.8,NaN
9,The Man with the Golden Gun,1974,Roger Moore,Guy Hamilton,334.0,27.7,NaN


## Cleaning column names

A list comprehension walks every column and replaces spaces with underscores. Consistent column names make later code (like `query`) much easier to write.

In [31]:
bond.columns = [column_name.replace(" ","_") for column_name in bond.columns]

In [32]:
bond.head()

,Film,Year,Actor,Director,Box_Office,Budget,Bond_Actor_Salary
0,Dr. No,1962,Sean Connery,Terence Young,448.8,7.0,0.6
1,From Russia with Love,1963,Sean Connery,Terence Young,543.8,12.6,1.6
2,Goldfinger,1964,Sean Connery,Guy Hamilton,820.4,18.6,3.2
3,Thunderball,1965,Sean Connery,Terence Young,848.1,41.9,4.7
4,Casino Royale,1967,David Niven,Ken Hughes,315.0,85.0,NaN


In [33]:
bond.query("Box_Office > 500")

,Film,Year,Actor,Director,Box_Office,Budget,Bond_Actor_Salary
1,From Russia with Love,1963,Sean Connery,Terence Young,543.8,12.6,1.6
2,Goldfinger,1964,Sean Connery,Guy Hamilton,820.4,18.6,3.2
3,Thunderball,1965,Sean Connery,Terence Young,848.1,41.9,4.7
5,You Only Live Twice,1967,Sean Connery,Lewis Gilbert,514.2,59.9,4.4
10,The Spy Who Loved Me,1977,Roger Moore,Lewis Gilbert,533.0,45.1,NaN
11,Moonraker,1979,Roger Moore,Lewis Gilbert,535.0,91.5,NaN
18,GoldenEye,1995,Pierce Brosnan,Martin Campbell,518.5,76.9,5.1
22,Casino Royale,2006,Daniel Craig,Martin Campbell,581.5,145.3,3.3
23,Quantum of Solace,2008,Daniel Craig,Marc Forster,514.2,181.4,8.1
24,Skyfall,2012,Daniel Craig,Sam Mendes,943.5,170.2,14.5


## `apply` — running a function on every value

`convert_to_millions` runs once per value, and the results are stored back. It's the escape hatch for logic too complex to vectorize.

In [34]:
def convert_to_millions(number):
    return str(number) + " Millions!"

In [35]:
bond["Box_Office"] = bond["Box_Office"].apply(convert_to_millions)
bond["Box_Office"]

0     448.8 Millions!
1     543.8 Millions!
2     820.4 Millions!
3     848.1 Millions!
4     315.0 Millions!
5     514.2 Millions!
6     291.5 Millions!
7     442.5 Millions!
8     460.3 Millions!
9     334.0 Millions!
10    533.0 Millions!
11    535.0 Millions!
12    449.4 Millions!
13    380.0 Millions!
14    373.8 Millions!
15    275.2 Millions!
16    313.5 Millions!
17    250.9 Millions!
18    518.5 Millions!
19    463.2 Millions!
20    439.5 Millions!
21    465.4 Millions!
22    581.5 Millions!
23    514.2 Millions!
24    943.5 Millions!
25    726.7 Millions!
Name: Box_Office, dtype: str

## Copy vs view — the silent bug factory

`directors = bond["Director"]` is a **view**; editing it may (silently) edit the original. `directors = bond["Director"].copy()` is a real copy — safe to modify. When in doubt, call `.copy()`.

In [36]:
directors = bond["Director"]

In [37]:
directors = bond["Director"].copy()

In [38]:
directors

0          Terence Young
1          Terence Young
2           Guy Hamilton
3          Terence Young
4             Ken Hughes
5          Lewis Gilbert
6          Peter R. Hunt
7           Guy Hamilton
8           Guy Hamilton
9           Guy Hamilton
10         Lewis Gilbert
11         Lewis Gilbert
12             John Glen
13        Irvin Kershner
14             John Glen
15             John Glen
16             John Glen
17             John Glen
18       Martin Campbell
19    Roger Spottiswoode
20         Michael Apted
21          Lee Tamahori
22       Martin Campbell
23          Marc Forster
24            Sam Mendes
25            Sam Mendes
Name: Director, dtype: str

In [39]:
directors["Sam Mendes"] = "Sit Sam Mendes"

In [40]:
directors

0                  Terence Young
1                  Terence Young
2                   Guy Hamilton
3                  Terence Young
4                     Ken Hughes
5                  Lewis Gilbert
6                  Peter R. Hunt
7                   Guy Hamilton
8                   Guy Hamilton
9                   Guy Hamilton
10                 Lewis Gilbert
11                 Lewis Gilbert
12                     John Glen
13                Irvin Kershner
14                     John Glen
15                     John Glen
16                     John Glen
17                     John Glen
18               Martin Campbell
19            Roger Spottiswoode
20                 Michael Apted
21                  Lee Tamahori
22               Martin Campbell
23                  Marc Forster
24                    Sam Mendes
25                    Sam Mendes
Sam Mendes        Sit Sam Mendes
Name: Director, dtype: str

In [41]:
bond

,Film,Year,Actor,Director,Box_Office,Budget,Bond_Actor_Salary
0,Dr. No,1962,Sean Connery,Terence Young,448.8 Millions!,7.0,0.6
1,From Russia with Love,1963,Sean Connery,Terence Young,543.8 Millions!,12.6,1.6
2,Goldfinger,1964,Sean Connery,Guy Hamilton,820.4 Millions!,18.6,3.2
3,Thunderball,1965,Sean Connery,Terence Young,848.1 Millions!,41.9,4.7
4,Casino Royale,1967,David Niven,Ken Hughes,315.0 Millions!,85.0,NaN
5,You Only Live Twice,1967,Sean Connery,Lewis Gilbert,514.2 Millions!,59.9,4.4
6,On Her Majesty's Secret Service,1969,George Lazenby,Peter R. Hunt,291.5 Millions!,37.3,0.6
7,Diamonds Are Forever,1971,Sean Connery,Guy Hamilton,442.5 Millions!,34.7,5.8
8,Live and Let Die,1973,Roger Moore,Guy Hamilton,460.3 Millions!,30.8,NaN
9,The Man with the Golden Gun,1974,Roger Moore,Guy Hamilton,334.0 Millions!,27.7,NaN
